In [1]:
import os
import json
import pandas as pd
import numpy as np
import string
from pathlib import Path
pd.set_option('display.max_colwidth', None)

In [2]:
import string
import pandas as pd
import re

def clean_turkish_chars(val):
    turkish_map = str.maketrans("çğıöşüÇĞİÖŞÜ", "cgiosuCGIOSU")
    if isinstance(val, list):
        return [v.translate(turkish_map) if isinstance(v, str) else v for v in val]
    elif isinstance(val, str):
        return val.translate(turkish_map)
    return val


with open('stop-words_tr.txt', encoding='utf-8') as f:
    stop_words = set(line.strip().lower() for line in f if line.strip())

def has_numbers(word):
    return any(char.isdigit() for char in word)

def is_unit(word):
    return len(word) <= 2

def split_compound_words(text):
    """Birleşik kelimelerdeki bağlaçları ayırır"""
    if not isinstance(text, str):
        return text
    text = re.sub(r'(\w+)ve(\w+)', r'\1 ve \2', text, flags=re.IGNORECASE)
    text = re.sub(r'(\w+)ile(\w+)', r'\1 ile \2', text, flags=re.IGNORECASE)
    
    return text

def separate_words(sent):
    if not isinstance(sent, str):
        return ""
    sent = split_compound_words(sent)
    sent = clean_turkish_chars(sent)
    sent = sent.translate(str.maketrans('', '', string.punctuation))
    sent = sent.lower()
    splitted = sent.split()
    
    filtered_words = [
        w for w in splitted
        if w not in stop_words and not has_numbers(w) and not is_unit(w) and w.strip()
    ]
    return ' '.join(filtered_words)

def clean_column(col):
    return col.astype(str).apply(separate_words)



def clean_folder_name(folder_name):
    """Klasör ismini temizle"""
    if not folder_name:
        return ""
    folder_name = folder_name.replace('-', ' ')
    folder_name = split_compound_words(folder_name)
    folder_name = clean_turkish_chars(folder_name)
    folder_name = folder_name.lower()
    splitted = folder_name.split()
    filtered_words = [
        w for w in splitted
        if w not in stop_words and not has_numbers(w) and not is_unit(w) and w.strip()
    ]
    return ' '.join(filtered_words)

In [3]:
# ==========================
# 1. Klasörlerden veriyi okuma
# ==========================
base_path = Path(r"D:\WEB_SCRAPING\hepsiburada\datav4")
json_files = list(base_path.rglob("*.json"))
data_list = []

for file in json_files:
    parts = file.relative_to(base_path).parts
    
    # Klasör isimlerini temizle
    super_category = clean_folder_name(parts[0]) if len(parts) > 0 else None
    category_1 = clean_folder_name(parts[1]) if len(parts) > 1 else None
    category_2 = clean_folder_name(parts[2]) if len(parts) > 2 else None
    
    with open(file, "r", encoding="utf-8") as f:
        try:
            content = json.load(f)
        except json.JSONDecodeError:
            print(f"Hatalı JSON: {file}")
            continue
    
    if isinstance(content, list):
        for item in content:
            item["super_category"] = super_category
            item["category_1"] = category_1
            item["category_2"] = category_2
            data_list.append(item)
    elif isinstance(content, dict):
        content["super_category"] = super_category
        content["category_1"] = category_1
        content["category_2"] = category_2
        data_list.append(content)

df = pd.DataFrame(data_list)
df = df.drop(['supercategory', 'category'], axis=1, errors='ignore')

In [4]:
df['title'] = clean_column(df['title'])
df['super_category'] = clean_column(df['super_category'])
df['category_1'] = clean_column(df['category_1'])
df['category_2'] = clean_column(df['category_2'])

def combine_strings(row):
    """String sütunları birleştir"""
    combined_parts = []
    for col in ['title', 'super_category', 'category_1', 'category_2']:
        val = row[col]
        if isinstance(val, str) and val.strip():
            combined_parts.append(val.strip())
    
    return ' '.join(combined_parts)

df['Bow'] = df.apply(combine_strings, axis=1)

In [5]:
# ==========================
# 2. Fiyat & Rating Temizleme
# ==========================
df['price'] = (
    df['price']
    .replace(['N/A', 'None', None], np.nan)
    .astype(str)
    .str.replace(' TL', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .replace('', np.nan)
    .astype(float)
)

df['rating'] = (
    df['rating']
    .replace('N/A', np.nan)
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

df['evaluation'] = (
    df['evaluation']
    .replace('N/A', np.nan)
    .astype(str)
    .str.extract(r'(\d+)')
    .astype(float)
)

In [6]:
# ==========================
# 3. Duplicate ürünleri silme
# ==========================
initial_count = len(df)
url_counts = df['product_url'].value_counts()
unique_urls = url_counts[url_counts == 1].index
df = df[df['product_url'].isin(unique_urls)].reset_index(drop=True)
removed_count = initial_count - len(df)
print(f"Silinen ürün sayısı: {removed_count}")

Silinen ürün sayısı: 27827


In [7]:
# ==========================
# 4. Weighted rating hesaplama
# ==========================
df['rating'] = df['rating'].fillna(0.0)
df['evaluation'] = df['evaluation'].fillna(0)

m = 3
C = df['rating'].mean()

def weighted_rating(row):
    v = row['evaluation']
    R = row['rating']
    return (v / (v + m)) * R + (m / (v + m)) * C

df['weighted_rating'] = df.apply(weighted_rating, axis=1)

bins_2 = [-1, 2, float('inf')]
labels_2 = [0, 1]

df['weighted_rating_class'] = pd.cut(df['weighted_rating'], bins=bins_2, labels=labels_2).astype(int)
rating_map = {0: "bad", 1: "good"}
df['weighted_rating_label'] = df['weighted_rating_class'].map(rating_map)

In [8]:
df.head()

,model,title,price,rating,evaluation,category_url,product_url,super_category,category_1,category_2,Bow,weighted_rating,weighted_rating_class,weighted_rating_label
0,Elfobaby,eglenceli kopuklu yengec banyo aktivite oyuncagi test raporlu,349.30,3.8,24.0,https://www.hepsiburada.com/aktivite-merkezi-c-80382017?sayfa=1,https://www.hepsiburada.com/eglenceli-kopuklu-yengec-banyo-aktivite-oyuncagi-pm-HBC00006VEGHP,anne bebek oyuncak,bebek aktivite eglence,aktivite merkezi,eglenceli kopuklu yengec banyo aktivite oyuncagi test raporlu anne bebek oyuncak bebek aktivite eglence aktivite merkezi,3.501070,1,good
1,Moniev,cocuk oyun etkinlik calisma masasi montessori sandalyeli,1495.25,4.7,116.0,https://www.hepsiburada.com/aktivite-merkezi-c-80382017?sayfa=1,https://www.hepsiburada.com/moniev-cocuk-oyun-etkinlik-calisma-masasi-montessori-sandalyeli-p-HBCV000046C332,anne bebek oyuncak,bebek aktivite eglence,aktivite merkezi,cocuk oyun etkinlik calisma masasi montessori sandalyeli anne bebek oyuncak bebek aktivite eglence aktivite merkezi,4.609487,1,good
2,Dynavica,renkli halkalar ice gecirme oyunu,339.00,0.0,0.0,https://www.hepsiburada.com/aktivite-merkezi-c-80382017?sayfa=1,https://adservice.hepsiburada.com/event/api/v1/track?event=bURVQWRvWVFNL1RvYk1aeXhNc3JLWEJWaElpYlMxWTQ4VWN2Y2s5ZVc0RDRWNC8zV2lQT0NVV1NZZUV1QTU4TjQ1aGlMNXhKa2RxN256MDlpLzVWT1VxWER1NldUOVlHWVlObERIb0kzdW1lTE9yWUMyR2o5UHRESkxVTWpyalhlclRzV1liTVJkMkRRb0ovcmR0QjdxeTVpSVRiYnV6enQ4aUMwdHZiSEdYSngwMzU0Szd1SkRUNjl1Q2laNlV5QlczWlJyaXdSRWVpN1NjNk1rYmdMMytaS0ZnN0ppQVhFM2pwd1lwWTFvUkgzd0xlK1QxejZ4OU95U3RCb0Y1bDJJaFVjQU5aUHR3OEU5bEVwQ2YwUzJod3Z4UHBVMk1zVGRlZUxuNE5weWg0L2tleUxsS3pwSUp2ZDJBSjk1YXdIOXdUR2I3Slc3WEtNWUw2NmdlTGVvd2hobVhFNjliQlovSzFhM1QwNWV4TlVHUFdMbkhpR3NiMk9DcSttbEdCejlJRXgyTWNVejF4RmdRNnE2M0FOZXViL25pQjRHNU9OVkw4MDloT3RzZkNKOGU1Zk5RR3J4dGxwQng2ay9xY0JsMEE5THJSWGZ1ZGhSWGh1L2V1anQ3YWkvK3pIb0dWUUZWaEQ4b2JvWGJnMDJFSGVnVG1ybUcyd2VtaTJ0OEtSOHNpSzlieDZVYzNHZWthZVd5T3ZHazVoUHVnb0RBTXNUaHpnNFdiOGYxdkF6ZTBVeWZDSzlxQ1cwTFo=&redirect=https%3A%2F%2Fwww.hepsiburada.com%2Frenkli-halkalar-ic-ice-gecirme-oyunu-p-HBCV000096CD8G%3Fmagaza%3DDYNAVICA&eventName=sp-click&platform=desktop,anne bebek oyuncak,bebek aktivite eglence,aktivite merkezi,renkli halkalar ice gecirme oyunu anne bebek oyuncak bebek aktivite eglence aktivite merkezi,1.109634,0,bad
3,Çocuk Akademi,stiker kitabim tasitlar,229.00,4.0,2.0,https://www.hepsiburada.com/aktivite-merkezi-c-80382017?sayfa=1,https://www.hepsiburada.com/cocuk-akademi-500-stiker-kitabim-tasitlar-pm-HBC00008M2ME0,anne bebek oyuncak,bebek aktivite eglence,aktivite merkezi,stiker kitabim tasitlar anne bebek oyuncak bebek aktivite eglence aktivite merkezi,2.265781,1,good
4,Babycim,donen kule,372.99,4.0,33.0,https://www.hepsiburada.com/aktivite-merkezi-c-80382017?sayfa=1,https://www.hepsiburada.com/babycim-donen-kule-pm-HBC000058DIUY,anne bebek oyuncak,bebek aktivite eglence,aktivite merkezi,donen kule anne bebek oyuncak bebek aktivite eglence aktivite merkezi,3.759136,1,good


In [9]:
# ==========================
# ==========================
# 6. Kaydet
# ==========================
output_path = base_path / "merged_cleaned_data.json"
df.to_json(output_path, orient="records", force_ascii=False)

print(f"{len(df)} satır kaydedildi -> {output_path}")

2907157 satır kaydedildi -> D:\WEB_SCRAPING\hepsiburada\datav4\merged_cleaned_data.json


In [10]:
df_new = pd.read_json(output_path)
df_new.head()

,model,title,price,rating,evaluation,category_url,product_url,super_category,category_1,category_2,Bow,weighted_rating,weighted_rating_class,weighted_rating_label
0,Elfobaby,eglenceli kopuklu yengec banyo aktivite oyuncagi test raporlu,349.30,3.8,24,https://www.hepsiburada.com/aktivite-merkezi-c-80382017?sayfa=1,https://www.hepsiburada.com/eglenceli-kopuklu-yengec-banyo-aktivite-oyuncagi-pm-HBC00006VEGHP,anne bebek oyuncak,bebek aktivite eglence,aktivite merkezi,eglenceli kopuklu yengec banyo aktivite oyuncagi test raporlu anne bebek oyuncak bebek aktivite eglence aktivite merkezi,3.501070,1,good
1,Moniev,cocuk oyun etkinlik calisma masasi montessori sandalyeli,1495.25,4.7,116,https://www.hepsiburada.com/aktivite-merkezi-c-80382017?sayfa=1,https://www.hepsiburada.com/moniev-cocuk-oyun-etkinlik-calisma-masasi-montessori-sandalyeli-p-HBCV000046C332,anne bebek oyuncak,bebek aktivite eglence,aktivite merkezi,cocuk oyun etkinlik calisma masasi montessori sandalyeli anne bebek oyuncak bebek aktivite eglence aktivite merkezi,4.609487,1,good
2,Dynavica,renkli halkalar ice gecirme oyunu,339.00,0.0,0,https://www.hepsiburada.com/aktivite-merkezi-c-80382017?sayfa=1,https://adservice.hepsiburada.com/event/api/v1/track?event=bURVQWRvWVFNL1RvYk1aeXhNc3JLWEJWaElpYlMxWTQ4VWN2Y2s5ZVc0RDRWNC8zV2lQT0NVV1NZZUV1QTU4TjQ1aGlMNXhKa2RxN256MDlpLzVWT1VxWER1NldUOVlHWVlObERIb0kzdW1lTE9yWUMyR2o5UHRESkxVTWpyalhlclRzV1liTVJkMkRRb0ovcmR0QjdxeTVpSVRiYnV6enQ4aUMwdHZiSEdYSngwMzU0Szd1SkRUNjl1Q2laNlV5QlczWlJyaXdSRWVpN1NjNk1rYmdMMytaS0ZnN0ppQVhFM2pwd1lwWTFvUkgzd0xlK1QxejZ4OU95U3RCb0Y1bDJJaFVjQU5aUHR3OEU5bEVwQ2YwUzJod3Z4UHBVMk1zVGRlZUxuNE5weWg0L2tleUxsS3pwSUp2ZDJBSjk1YXdIOXdUR2I3Slc3WEtNWUw2NmdlTGVvd2hobVhFNjliQlovSzFhM1QwNWV4TlVHUFdMbkhpR3NiMk9DcSttbEdCejlJRXgyTWNVejF4RmdRNnE2M0FOZXViL25pQjRHNU9OVkw4MDloT3RzZkNKOGU1Zk5RR3J4dGxwQng2ay9xY0JsMEE5THJSWGZ1ZGhSWGh1L2V1anQ3YWkvK3pIb0dWUUZWaEQ4b2JvWGJnMDJFSGVnVG1ybUcyd2VtaTJ0OEtSOHNpSzlieDZVYzNHZWthZVd5T3ZHazVoUHVnb0RBTXNUaHpnNFdiOGYxdkF6ZTBVeWZDSzlxQ1cwTFo=&redirect=https%3A%2F%2Fwww.hepsiburada.com%2Frenkli-halkalar-ic-ice-gecirme-oyunu-p-HBCV000096CD8G%3Fmagaza%3DDYNAVICA&eventName=sp-click&platform=desktop,anne bebek oyuncak,bebek aktivite eglence,aktivite merkezi,renkli halkalar ice gecirme oyunu anne bebek oyuncak bebek aktivite eglence aktivite merkezi,1.109634,0,bad
3,Çocuk Akademi,stiker kitabim tasitlar,229.00,4.0,2,https://www.hepsiburada.com/aktivite-merkezi-c-80382017?sayfa=1,https://www.hepsiburada.com/cocuk-akademi-500-stiker-kitabim-tasitlar-pm-HBC00008M2ME0,anne bebek oyuncak,bebek aktivite eglence,aktivite merkezi,stiker kitabim tasitlar anne bebek oyuncak bebek aktivite eglence aktivite merkezi,2.265781,1,good
4,Babycim,donen kule,372.99,4.0,33,https://www.hepsiburada.com/aktivite-merkezi-c-80382017?sayfa=1,https://www.hepsiburada.com/babycim-donen-kule-pm-HBC000058DIUY,anne bebek oyuncak,bebek aktivite eglence,aktivite merkezi,donen kule anne bebek oyuncak bebek aktivite eglence aktivite merkezi,3.759136,1,good


In [17]:
print(df[(df['super_category'] == 'elektronik') & (df['category_1'] == 'bilgisayar sistemleri ekipmanlari')]['category_2'].value_counts())

category_2
aksesuarlar                   43025
yazici                        23321
cevre birimleri               21764
bilgisayar senleri            13935
modem                         13390
bilgisayarlar                  9715
bilgisayar yedek parcalari     6809
veri depolama                  5819
oyuncu ozel                    1494
yazilim urunleri               1339
tablet                         1161
ses kayit cihazlari             842
Name: count, dtype: int64


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2907157 entries, 0 to 2907156
Data columns (total 14 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   model                  object 
 1   title                  object 
 2   price                  float64
 3   rating                 float64
 4   evaluation             float64
 5   category_url           object 
 6   product_url            object 
 7   super_category         object 
 8   category_1             object 
 9   category_2             object 
 10  Bow                    object 
 11  weighted_rating        float64
 12  weighted_rating_class  int32  
 13  weighted_rating_label  object 
dtypes: float64(4), int32(1), object(9)
memory usage: 299.4+ MB
